Config

In [29]:
CONFIG = {
    # representation
    "window_size": 5,
    "alphabet_size": 3,          # next SAX symbol in {a,b,c}
    "step_symbol": 1,            # predict 1 step ahead in SYMBOL space
    "sax_strategy": "uniform",

    # data
    "n_points": 320,
    "noise_std": 0.10,
    "seed_data": 123,

    # split (mitigate missing-class issue)
    "train_frac": 0.8,
    "split_mode": "middle_block",  # "tail" or "middle_block" or "rolling"
    "min_per_class_test": 3,        # require >= this many per class in test
    "max_split_tries": 500,

    # models
    "seed_model": 12345,
    "reps_vqc": 1,
    "maxiter_vqc": 50,

    # qsvc fixed reps=1 (keep KISS)
}


Dataset (window -> next SAX symbol)

In [30]:
import numpy as np
from pyts.approximation import SymbolicAggregateApproximation

def make_dataset_next_sax_symbol(
    *,
    window_size,
    alphabet_size,
    n_points,
    noise_std,
    seed,
    step_symbol=1,
    sax_strategy="uniform",
    plot=False,
):
    rng = np.random.default_rng(seed)

    t = np.linspace(0, 6 * np.pi, n_points)
    ts = np.sin(t) + rng.normal(0.0, noise_std, size=n_points)

    if plot:
        import matplotlib.pyplot as plt
        plt.plot(t, ts)
        plt.title("Noisy Sine Wave Time Series")
        plt.xlabel("Time")
        plt.ylabel("Value")
        plt.show()

    sax = SymbolicAggregateApproximation(n_bins=alphabet_size, strategy=sax_strategy)
    sax_seq = sax.fit_transform(ts.reshape(1, -1))[0]  # (n_points,)

    X_sax, y = [], []
    last_start = n_points - window_size - step_symbol
    for i in range(last_start):
        X_sax.append(sax_seq[i : i + window_size])
        y.append(sax_seq[i + window_size + step_symbol - 1])

    return np.asarray(X_sax), np.asarray(y), sax_seq


Utilities

In [31]:
from collections import Counter
import numpy as np
from sklearn.preprocessing import LabelEncoder

def sax_word_stats(X_sax, top_k=5):
    X_sax = np.asarray(X_sax)
    words = ["".join(row.astype(str).tolist()) for row in X_sax]
    uniq = len(set(words))
    return {
        "n": len(words),
        "unique": uniq,
        "unique_ratio": uniq / max(1, len(words)),
        "top_words": Counter(words).most_common(top_k),
    }

def sax_to_angles(X_sax, eps=1e-3):
    X_sax = np.asarray(X_sax).astype(str)
    alphabet = sorted(np.unique(X_sax.reshape(-1)).tolist())
    k = len(alphabet)
    if k < 2:
        raise ValueError(f"Alphabet too small ({k}).")

    angles = np.linspace(eps, np.pi - eps, k)
    mapping = {sym: ang for sym, ang in zip(alphabet, angles)}

    X = np.empty(X_sax.shape, dtype=float)
    for sym in np.unique(X_sax):
        if sym not in mapping:
            raise ValueError(f"Unknown symbol '{sym}'.")
        X[X_sax == sym] = mapping[sym]
    return X, mapping

def encode_labels(y_symbols):
    y_symbols = np.asarray(y_symbols).astype(str)
    le = LabelEncoder()
    return le.fit_transform(y_symbols), le

def count_labels(y_enc, le):
    counts = np.bincount(y_enc, minlength=len(le.classes_))
    return dict(zip(le.classes_, counts))


Split strategies

In [32]:
import numpy as np

def split_tail(X, y, train_frac=0.8):
    n = len(X)
    split = int(train_frac * n)
    if split <= 1 or split >= n:
        raise ValueError("Bad split. Adjust train_frac or n.")
    return X[:split], X[split:], y[:split], y[split:], ("tail", split, n)

def split_middle_block_with_coverage(
    X, y, *, train_frac=0.8, min_per_class=3, max_tries=500
):
    """
    Time-ordered split where TEST is a contiguous middle block.
    Ensures TEST contains >= min_per_class examples of every class present in y.
    """
    X = np.asarray(X)
    y = np.asarray(y)
    n = len(X)

    classes = np.unique(y)
    k = len(classes)

    test_len = max(2, int(round((1.0 - train_frac) * n)))
    test_len = max(test_len, k * min_per_class)

    if test_len >= n - 2:
        raise ValueError("Test block too large for dataset. Increase n_points or reduce min_per_class/test size.")

    lo = max(0, n // 10)
    hi = min(n - test_len, n - n // 10)
    if hi <= lo:
        lo, hi = 0, n - test_len

    candidates = np.linspace(lo, hi, num=min(max_tries, 80), dtype=int).tolist()
    if len(candidates) < max_tries:
        candidates += np.linspace(lo, hi, num=max_tries - len(candidates), dtype=int).tolist()

    for start in candidates[:max_tries]:
        end = start + test_len
        y_test = y[start:end]

        ok = True
        for c in classes:
            if np.sum(y_test == c) < min_per_class:
                ok = False
                break
        if not ok:
            continue

        test_idx = np.arange(start, end)
        train_idx = np.setdiff1d(np.arange(n), test_idx, assume_unique=True)

        return X[train_idx], X[test_idx], y[train_idx], y[test_idx], ("middle", start, end)

    raise ValueError("Could not find middle test block with required class coverage.")

def rolling_origin_splits(X, y, *, train_min=150, test_len=60, step=60):
    """
    Produces multiple (train, test) contiguous splits (time-ordered) for robust evaluation.
    Use when any single split is unstable.
    """
    X = np.asarray(X); y = np.asarray(y)
    n = len(X)
    splits = []
    start = train_min
    while start + test_len <= n:
        X_train, y_train = X[:start], y[:start]
        print("train_counts:", dict(zip(le.classes_, np.bincount(y_train, minlength=len(le.classes_)))))
        print("test_counts: ", dict(zip(le.classes_, np.bincount(y_test,  minlength=len(le.classes_)))))
        X_test, y_test = X[start:start+test_len], y[start:start+test_len]
        splits.append((X_train, X_test, y_train, y_test, ("rolling", start, start+test_len)))
        start += step
    if not splits:
        raise ValueError("No rolling splits possible. Increase n_points or reduce train_min/test_len.")
    return splits

Models (baselines + QSVC + VQC; QSVC reps fixed to 1)

In [33]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

from qiskit.circuit.library import real_amplitudes, zz_feature_map
from qiskit_machine_learning.utils import algorithm_globals
from qiskit_machine_learning.optimizers import COBYLA
from qiskit_machine_learning.algorithms.classifiers import VQC

from qiskit_machine_learning.algorithms import QSVC
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit.primitives import StatevectorSampler as Sampler

def eval_preds(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    f1m = f1_score(y_true, y_pred, average="macro")
    print(f"{name}: acc={acc:.3f} macroF1={f1m:.3f}")
    print(confusion_matrix(y_true, y_pred))
    return acc, f1m

def run_classical_baselines(X_train, X_test, y_train, y_test):
    lin = make_pipeline(StandardScaler(), LinearSVC())
    rbf = make_pipeline(StandardScaler(), SVC(kernel="rbf", gamma="scale"))

    lin.fit(X_train, y_train)
    rbf.fit(X_train, y_train)

    out = {}
    out["LinearSVC"] = eval_preds("LinearSVC", y_test, lin.predict(X_test))[0]
    out["RBF-SVC"]   = eval_preds("RBF-SVC",   y_test, rbf.predict(X_test))[0]
    return out

def run_qsvc(X_train, X_test, y_train, y_test, seed=12345, reps=1):
    algorithm_globals.random_seed = seed

    zz_fm = zz_feature_map(feature_dimension=X_train.shape[1], reps=reps, entanglement="full")
    sampler = Sampler()
    fidelity = ComputeUncompute(sampler=sampler)
    qkernel = FidelityQuantumKernel(fidelity=fidelity, feature_map=zz_fm)

    model = QSVC(quantum_kernel=qkernel)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    acc, _ = eval_preds(f"QSVC(reps={reps})", y_test, y_pred)
    return acc

def run_vqc(X_train, X_test, y_train, y_test, seed=12345, fm_reps=1, maxiter=50):
    algorithm_globals.random_seed = seed

    fm = zz_feature_map(feature_dimension=X_train.shape[1], reps=fm_reps, entanglement="full")
    ansatz = real_amplitudes(num_qubits=X_train.shape[1], reps=1, entanglement="full")

    optimizer = COBYLA(maxiter=maxiter)
    sampler = Sampler()

    model = VQC(feature_map=fm, ansatz=ansatz, optimizer=optimizer, sampler=sampler)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    acc, _ = eval_preds(f"VQC(fm_reps={fm_reps})", y_test, y_pred)
    return acc


In [34]:
def oversample_to_min_count(X, y, min_count, seed=0):
    rng = np.random.default_rng(seed)
    X_parts, y_parts = [X], [y]
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        if len(idx) < min_count:
            extra = rng.choice(idx, size=(min_count - len(idx)), replace=True)
            X_parts.append(X[extra])
            y_parts.append(y[extra])
    return np.vstack(X_parts), np.concatenate(y_parts)


Report

In [35]:
def run_report_next_symbol(config):
    # 1) dataset (next SAX symbol)
    X_sax, y_sym, _ = make_dataset_next_sax_symbol(
        window_size=config["window_size"],
        alphabet_size=config["alphabet_size"],
        n_points=config["n_points"],
        noise_std=config["noise_std"],
        seed=config["seed_data"],
        step_symbol=config["step_symbol"],
        sax_strategy=config["sax_strategy"],
        plot=False,
    )

    # 2) representation diagnostics
    stats = sax_word_stats(X_sax)
    y_enc, le = encode_labels(y_sym)

    print("\n=== RUN (next SAX symbol) ===")
    print(f"window={config['window_size']} alphabet={config['alphabet_size']} step={config['step_symbol']} strategy={config['sax_strategy']}")
    print(f"n={len(y_enc)} unique_ratio={stats['unique_ratio']:.3f} top_words={stats['top_words']}")

    # 3) features
    X_angles, _ = sax_to_angles(X_sax)

    # 4) choose split mode
    mode = config["split_mode"]
    if mode == "tail":
        splits = [( *split_tail(X_angles, y_enc, train_frac=config["train_frac"]), )]
    elif mode == "middle_block":
        X_tr, X_te, y_tr, y_te, seg = split_middle_block_with_coverage(
            X_angles, y_enc,
            train_frac=config["train_frac"],
            min_per_class=config["min_per_class_test"],
            max_tries=config["max_split_tries"],
        )
        splits = [(X_tr, X_te, y_tr, y_te, seg)]
    elif mode == "rolling":
        # rolling evaluation: average results across splits
        splits = rolling_origin_splits(
            X_angles, y_enc,
            train_min=int(config["train_frac"] * len(X_angles)),
            test_len=max(30, int(0.2 * len(X_angles))),
            step=max(30, int(0.2 * len(X_angles))),
        )
    else:
        raise ValueError(f"Unknown split_mode: {mode}")

    # 5) run models (possibly multiple splits)
    results = []
    for X_train, X_test, y_train, y_test, seg in splits:
        test_counts = count_labels(y_test, le)
        majority = max(test_counts.values()) / max(1, len(y_test))
        random_base = 1.0 / len(le.classes_)

        print("\n--- split:", seg, "---")
        print("test_counts:", test_counts, "| majority:", round(majority, 3), "| random:", round(random_base, 3))

        base = run_classical_baselines(X_train, X_test, y_train, y_test)
        # oversample TRAIN for VQC only
        Xtr_os, ytr_os = oversample_to_min_count(X_train, y_train, min_count=100, seed=config["seed_model"])
        vqc_acc = run_vqc(
        Xtr_os, X_test, ytr_os, y_test,
        seed=config["seed_model"],
        fm_reps=config["reps_vqc"],
        maxiter=config["maxiter_vqc"],
        )

        qsvc_acc = run_qsvc(
            X_train, X_test, y_train, y_test,
            seed=config["seed_model"],
            reps=1,
        )

        results.append({
            "split": seg,
            "unique_ratio": stats["unique_ratio"],
            "LinearSVC": base["LinearSVC"],
            "RBF-SVC": base["RBF-SVC"],
            "VQC": vqc_acc,
            "QSVC": qsvc_acc,
            "test_counts": test_counts,
        })

    # 6) compact summary (mean if rolling)
    if len(results) > 1:
        def mean_key(k): return float(np.mean([r[k] for r in results]))
        print("\nSUMMARY (mean over splits) | "
              f"uniq={stats['unique_ratio']:.3f} | "
              f"Linear={mean_key('LinearSVC'):.3f} | RBF={mean_key('RBF-SVC'):.3f} | "
              f"VQC={mean_key('VQC'):.3f} | QSVC={mean_key('QSVC'):.3f}")
    else:
        r = results[0]
        print("\nSUMMARY | "
              f"uniq={r['unique_ratio']:.3f} | "
              f"Linear={r['LinearSVC']:.3f} | RBF={r['RBF-SVC']:.3f} | "
              f"VQC={r['VQC']:.3f} | QSVC={r['QSVC']:.3f}")

    return {"config": dict(config), "labels": list(le.classes_), "stats": stats, "results": results}

out = run_report_next_symbol(CONFIG)


No gradient function provided, creating a gradient function. If your Sampler requires transpilation, please provide a pass manager.



=== RUN (next SAX symbol) ===
window=5 alphabet=3 step=1 strategy=uniform
n=314 unique_ratio=0.099 top_words=[('ccccc', 102), ('aaaaa', 102), ('bbbbb', 46), ('bbbbc', 4), ('cbbbb', 4)]

--- split: ('middle', 31, 94) ---
test_counts: {np.str_('a'): np.int64(39), np.str_('b'): np.int64(12), np.str_('c'): np.int64(12)} | majority: 0.619 | random: 0.333
LinearSVC: acc=0.937 macroF1=0.898
[[38  1  0]
 [ 0  9  3]
 [ 0  0 12]]
RBF-SVC: acc=0.937 macroF1=0.900
[[38  1  0]
 [ 0 10  2]
 [ 0  1 11]]
VQC(fm_reps=1): acc=0.746 macroF1=0.515
[[35  4  0]
 [ 0 12  0]
 [ 0 12  0]]
QSVC(reps=1): acc=0.921 macroF1=0.897
[[36  3  0]
 [ 0 11  1]
 [ 0  1 11]]

SUMMARY | uniq=0.099 | Linear=0.937 | RBF=0.937 | VQC=0.746 | QSVC=0.921
